# 🛠️ 03. Limpieza y Transformación de Datos (ETL) - Banco

## 🎯 Objetivo de este Notebook
Basado en los hallazgos de nuestro Análisis Exploratorio (Notebook 01), sabemos que el archivo `bank-additional.csv` tiene varios problemas estructurales: formatos de fecha incorrectos, números guardados como texto (por el uso de la coma decimal), mayúsculas inconsistentes y valores 'unknown'.

En lugar de crear un código espagueti e interminable, **aplicaré mi módulo personalizado `sp_lim.py`**, donde he programado una serie de funciones de limpieza estandarizadas. Esto me permite mantener el notebook limpio, legible y enfocado en el proceso de negocio.

**Pasos de la canalización (Pipeline):**
1. Estandarización general (Titulos en `snake_case` y textos normalizados).
2. Correcciones específicas (Fechas, IDs y valores 'unknown').
3. Limpieza numérica (Parseo de strings a floats).
4. Exportación a la capa *Silver* (Datos Procesados).


In [1]:
# -------------------------------------------------------------------------
# 1. PREPARACIÓN DEL ENTORNO Y CARGA DE DATOS
# -------------------------------------------------------------------------

# Configuración para que Jupyter recargue mis módulos externos automáticamente
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import sys

# Configuro la visualización para que no oculte columnas
pd.set_option('display.max_columns', None)

# Conecto con la carpeta raíz para poder importar mis módulos 'src'
sys.path.append(os.path.abspath(".."))
from src import sp_lim as sl
from src import sp_eda as se

# Carga de datos originales y creación de la copia de trabajo
ruta_raw = '../data/bank-additional.csv'
df_original = pd.read_csv(ruta_raw)
df_banco = df_original.copy()

print(f"✅ Entorno configurado. Datos cargados en memoria. Dimensiones: {df_banco.shape}")
display(df_banco.head(2))


✅ Entorno configurado. Datos cargados en memoria. Dimensiones: (43000, 24)


,Unnamed: 0,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_
0,0,NaN,housemaid,MARRIED,basic.4y,0.0,0.0,0.0,telephone,261,1,999,0,NONEXISTENT,1.1,"93,994","-36,4","4,857",5191,no,2-agosto-2019,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079
1,1,57.0,services,MARRIED,high.school,NaN,0.0,0.0,telephone,149,1,999,0,NONEXISTENT,1.1,"93,994","-36,4",NaN,5191,no,14-septiembre-2016,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097


## 🧹 Paso 1: Estandarización General
El primer paso para evitar errores de programación es asegurar que todos los datos tengan un formato predecible. 
Usaré mi función `limpiar_titulos` para pasar todas las columnas a formato `snake_case` (minúsculas y guiones bajos). Luego, aplicaré `limpiar_texto` a todas las columnas de tipo `object` (salvo identificadores y fechas) para estandarizar categorías, eliminar mayúsculas y quitar tildes/eñes.


In [2]:
# -------------------------------------------------------------------------
# 2. ESTANDARIZACIÓN DE TÍTULOS Y TEXTOS
# -------------------------------------------------------------------------

# 1. Arreglar títulos de columnas (ej: 'emp.var.rate' -> 'emp_var_rate')
sl.limpiar_titulos(df_banco)

# 2. Estandarizar contenido de texto
# Protegemos 'id_' (clave primaria) y 'date' (para no alterar los meses)
columnas_intocables = ['id_', 'date']
sl.limpiar_texto(df_banco, columnas_a_ignorar=columnas_intocables)

print("✅ Estandarización de texto completada. Vista previa:")
display(df_banco.head(3))


✅ Nombres de columnas estandarizados.
🧹 Limpiando contenido de 10 columnas...
✅ Textos estandarizados (sin ñ, espacios, puntos ni guiones).
✅ Estandarización de texto completada. Vista previa:


,unnamed:_0,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,date,latitude,longitude,id_
0,0,NaN,housemaid,married,basic_4y,0.0,0.0,0.0,telephone,261,1,999,0,nonexistent,1.1,"93,994","_36,4","4,857",5191,no,2-agosto-2019,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079
1,1,57.0,services,married,high_school,NaN,0.0,0.0,telephone,149,1,999,0,nonexistent,1.1,"93,994","_36,4",NaN,5191,no,14-septiembre-2016,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097
2,2,37.0,services,married,high_school,0.0,1.0,0.0,telephone,226,1,999,0,nonexistent,1.1,"93,994","_36,4","4,857",5191,no,15-febrero-2019,34.939,-94.847,3f9f49b5-e410-4948-bf6e-f9244f04918b


## 🎯 Paso 2: Correcciones Específicas de Negocio
Ahora que la base textual es homogénea, aplico reglas de negocio:
1. **Renombrar la PK:** Renombro la confusa columna `id_` a `ID` (Clave Primaria).
2. **Homogeneización del idioma:** La limpieza anterior pasó todo a minúsculas, así que aprovecho para buscar los registros `'unknown'` y traducirlos a `'desconocido'`.
3. **Conversión de Fechas:** Las fechas venían como texto con los meses en español (*'1-mayo-2016'*). Mi función `arreglar_fecha` mapea los meses a números y lo convierte a un tipo `datetime64` real de Pandas.


In [3]:
# -------------------------------------------------------------------------
# 3. RENOMBRADO, TRADUCCIÓN Y PARSEO DE FECHAS
# -------------------------------------------------------------------------

# 1. Renombrar clave primaria
sl.cambiar_nombres(df_banco, {'id_': 'ID'})

# 2. Gestionar valores 'unknown' en variables categóricas
columnas_con_unknown = ['job', 'marital', 'education', 'default', 'housing', 'loan']

for col in columnas_con_unknown:
    sl.reemplazar_valor(df_banco, col, 'unknown', 'desconocido')

# 3. Parsear la columna de fecha
sl.arreglar_fecha(df_banco, 'date')

print("✅ Reglas específicas aplicadas. Verificación de tipos de datos:")
print(df_banco[['ID', 'date']].dtypes)
display(df_banco.head(3))


✅ Se han renombrado las columnas: ['id_']
🔄 En columna 'job': se han cambiado 0 veces 'unknown' por 'desconocido'.
🔄 En columna 'marital': se han cambiado 0 veces 'unknown' por 'desconocido'.
🔄 En columna 'education': se han cambiado 0 veces 'unknown' por 'desconocido'.
🔄 En columna 'default': se han cambiado 0 veces 'unknown' por 'desconocido'.
🔄 En columna 'housing': se han cambiado 0 veces 'unknown' por 'desconocido'.
🔄 En columna 'loan': se han cambiado 0 veces 'unknown' por 'desconocido'.
📅 Columna 'date' convertida a fecha correctamente.
✅ Reglas específicas aplicadas. Verificación de tipos de datos:
ID              object
date    datetime64[ns]
dtype: object


,unnamed:_0,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,date,latitude,longitude,ID
0,0,NaN,housemaid,married,basic_4y,0.0,0.0,0.0,telephone,261,1,999,0,nonexistent,1.1,"93,994","_36,4","4,857",5191,no,2019-08-02,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079
1,1,57.0,services,married,high_school,NaN,0.0,0.0,telephone,149,1,999,0,nonexistent,1.1,"93,994","_36,4",NaN,5191,no,2016-09-14,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097
2,2,37.0,services,married,high_school,0.0,1.0,0.0,telephone,226,1,999,0,nonexistent,1.1,"93,994","_36,4","4,857",5191,no,2019-02-15,34.939,-94.847,3f9f49b5-e410-4948-bf6e-f9244f04918b


## 🔢 Paso 3: Corrección de Formatos Numéricos
En el EDA detecté que cinco columnas numéricas (`emp_var_rate`, `cons_price_idx`, etc.) se leían como texto (Object) porque utilizaban una coma `,` como separador decimal. Mi función iterará sobre ellas, cambiará la coma por un punto y las parseará a `float`. Por último, aplicaré una purga de filas totalmente duplicadas por higiene básica.

> ⚠️ **Nota analítica importante:** Al ejecutar este proceso, he detectado que la columna `cons_conf_idx` ha quedado vacía (0 no-nulos). Esto ocurre porque originalmente contenía valores negativos (ej: `-36,4`), y en el Paso 1, mi función general de limpieza de texto convirtió los guiones medios en guiones bajos (`_36,4`). Al intentar pasarlo a número, Pandas lo interpreta como error y lo fuerza a nulo (`coerce`). 
He decidido dejarlo así y documentarlo, ya que esta variable (Índice de Confianza del Consumidor) no es prioritaria para mi análisis actual.


In [4]:
# -------------------------------------------------------------------------
# 4. PARSEO DE NÚMEROS Y DUPLICADOS
# -------------------------------------------------------------------------

# Columnas económicas afectadas por la coma decimal
cols_numericas = ['emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed']

# Llamada al módulo para forzar el casteo numérico
sl.limpiar_numeros(df_banco, cols_numericas)

# Eliminación de duplicados absolutos
sl.eliminar_duplicados(df_banco)

# Verificación de la anomalía documentada en cons_conf_idx
print("\n--- INFO FINAL ---")
df_banco.info()


🔢 Arreglando formato numérico en: ['emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed']
✅ Números convertidos.
✅ No se encontraron duplicados.

--- INFO FINAL ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43000 entries, 0 to 42999
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   unnamed:_0      43000 non-null  int64         
 1   age             37880 non-null  float64       
 2   job             42655 non-null  object        
 3   marital         42915 non-null  object        
 4   education       41193 non-null  object        
 5   default         34019 non-null  float64       
 6   housing         41974 non-null  float64       
 7   loan            41974 non-null  float64       
 8   contact         43000 non-null  object        
 9   duration        43000 non-null  int64         
 10  campaign        43000 non-null  int64         
 11  pdays           43000

## 💾 Paso 4: Cierre y Exportación
El dataset del banco ya está limpio. El último paso es deshacernos de columnas basura que se generaron al importar/exportar originalmente (como `unnamed:_0`) y guardar este DataFrame en un archivo limpio dentro de nuestra carpeta de `procesados`. 
¡Listo para unirse con los datos del cliente!


In [5]:
# -------------------------------------------------------------------------
# 5. ELIMINACIÓN DE COLUMNAS BASURA Y GUARDADO
# -------------------------------------------------------------------------

# Borrar el índice original exportado por error
sl.eliminar_columnas(df_banco, ['unnamed:_0'])

# Definir la ruta de destino (capa procesada)
ruta_guardado = '../data/procesados/bank_clean.csv'

# Guardar a CSV sin índice numérico adicional
df_banco.to_csv(ruta_guardado, index=False)

print(f"✅ ¡TRABAJO TERMINADO!")
print(f"💾 Archivo limpio y formateado guardado correctamente en: {ruta_guardado}")

# Vista definitiva del set de datos preparado
print("\n--- VISTA PREVIA DEL ARCHIVO LIMPIO ---")
display(df_banco.head(3))


🗑️ Columnas eliminadas: ['unnamed:_0']
✅ ¡TRABAJO TERMINADO!
💾 Archivo limpio y formateado guardado correctamente en: ../data/procesados/bank_clean.csv

--- VISTA PREVIA DEL ARCHIVO LIMPIO ---


,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,date,latitude,longitude,ID
0,NaN,housemaid,married,basic_4y,0.0,0.0,0.0,telephone,261,1,999,0,nonexistent,1.1,93.994,NaN,4.857,5191.0,no,2019-08-02,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079
1,57.0,services,married,high_school,NaN,0.0,0.0,telephone,149,1,999,0,nonexistent,1.1,93.994,NaN,NaN,5191.0,no,2016-09-14,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097
2,37.0,services,married,high_school,0.0,1.0,0.0,telephone,226,1,999,0,nonexistent,1.1,93.994,NaN,4.857,5191.0,no,2019-02-15,34.939,-94.847,3f9f49b5-e410-4948-bf6e-f9244f04918b
